# Customer Churn Risk Analysis
### SaaS Product – Intercom User Data (2023–2026)

**Objective:** Identify users at risk of churning (unsubscribing or hard-bouncing) based on engagement signals, and build a risk-scoring model to help the CSR team prioritise outreach.

**Dataset:** 2,286 Intercom users exported May 2026  
**Author:** Kristi Panta  



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120


## 1. Load & Preview Data

In [ ]:
df_raw = pd.read_csv('data/Bubbles_Users_1074734_export_2026-05-27_13_04.csv')
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)


## 2. Exploratory Data Analysis
### 2a. Data Quality Overview

In [ ]:
# Show fill-rate for all columns
fill_rate = (df_raw.notna().sum() / len(df_raw) * 100).sort_values(ascending=False)
fill_df = fill_rate.reset_index()
fill_df.columns = ['Column', 'Fill Rate (%)']

fig, ax = plt.subplots(figsize=(10, 14))
colors = ['#4C9BE8' if v >= 50 else '#E87B4C' for v in fill_df['Fill Rate (%)']]
ax.barh(fill_df['Column'], fill_df['Fill Rate (%)'], color=colors)
ax.axvline(50, color='red', linestyle='--', linewidth=1, label='50% threshold')
ax.set_xlabel('Fill Rate (%)')
ax.set_title('Column Fill Rates – Intercom Export', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('data/fill_rates.png', bbox_inches='tight')
plt.show()
print("\nColumns with >50% fill rate:")
print(fill_df[fill_df['Fill Rate (%)'] >= 50]['Column'].tolist())


## 3. Feature Engineering

In [ ]:
df = df_raw.copy()

# ── Parse dates
for col in ['First Seen (PDT)', 'Last contacted (PDT)', 'Last heard from (PDT)']:
    df[col] = pd.to_datetime(df[col])

REF_DATE = pd.Timestamp('2026-05-27')

# ── Time-based features
df['days_since_first_seen']      = (REF_DATE - df['First Seen (PDT)']).dt.days
df['days_since_last_contacted']  = (REF_DATE - df['Last contacted (PDT)']).dt.days.fillna(999)
df['engagement_window_days']     = (df['Last contacted (PDT)'] - df['First Seen (PDT)']).dt.days.fillna(0)

# ── Behavioural features
df['has_been_heard_from'] = df['Last heard from (PDT)'].notna().astype(int)
df['web_sessions']        = df['Web sessions'].fillna(0).astype(int)

# ── Email features
df['email_domain']    = df['Email'].str.split('@').str[-1]
df['is_free_email']   = df['email_domain'].isin(
    ['gmail.com','hotmail.com','outlook.com','yahoo.com','icloud.com']).astype(int)

# ── Churn label
# A user is "churned" if they unsubscribed OR their email hard-bounced
df['churned'] = (
    (df['Unsubscribed from Emails'] == True) |
    (df['Has hard bounced']         == True)
).astype(int)

print("Churn label distribution:")
print(df['churned'].value_counts().rename({0:'Active', 1:'Churned'}))
print(f"\nOverall churn rate: {df['churned'].mean():.1%}")


### 2b. Churn Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
counts = df['churned'].value_counts().rename({0: 'Active', 1: 'Churned'})
axes[0].bar(counts.index, counts.values, color=['#4C9BE8','#E87B4C'], width=0.5, edgecolor='white')
axes[0].set_title('User Count by Churn Status', fontweight='bold')
axes[0].set_ylabel('Users')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4C9BE8','#E87B4C'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Rate', fontweight='bold')

plt.suptitle('Churn Label Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/churn_distribution.png', bbox_inches='tight')
plt.show()


### 2c. Churn Rate by Email Type (Free vs Business)

In [ ]:
email_churn = df.groupby('is_free_email')['churned'].agg(['mean','count']).reset_index()
email_churn['label'] = email_churn['is_free_email'].map({0:'Business email', 1:'Free email (Gmail etc.)'})
email_churn['churn_pct'] = email_churn['mean'] * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(email_churn['label'], email_churn['churn_pct'],
              color=['#4C9BE8','#E87B4C'], width=0.45, edgecolor='white')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate: Free Email vs Business Email', fontweight='bold')
for bar, pct, n in zip(bars, email_churn['churn_pct'], email_churn['count']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1,
            f'{pct:.1f}%\n(n={n:,})', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('data/churn_by_email_type.png', bbox_inches='tight')
plt.show()

print("Insight: Free email users churn at 2× the rate of business email users.")


### 2d. Churn Rate by User Response Behaviour

In [ ]:
heard_churn = df.groupby('has_been_heard_from')['churned'].agg(['mean','count']).reset_index()
heard_churn['label'] = heard_churn['has_been_heard_from'].map({0:'Never replied', 1:'Replied at least once'})
heard_churn['churn_pct'] = heard_churn['mean'] * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(heard_churn['label'], heard_churn['churn_pct'],
              color=['#E87B4C','#4C9BE8'], width=0.45, edgecolor='white')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Whether User Ever Replied', fontweight='bold')
for bar, pct, n in zip(bars, heard_churn['churn_pct'], heard_churn['count']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1,
            f'{pct:.1f}%\n(n={n:,})', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('data/churn_by_response.png', bbox_inches='tight')
plt.show()

print("Insight: Not a single user who replied ever churned — response = strongest retention signal.")


### 2e. Top Email Domains Among Churned Users

In [ ]:
churned_domains = (df[df['churned']==1]['email_domain']
                   .value_counts()
                   .head(12)
                   .reset_index())
churned_domains.columns = ['domain', 'churned_users']

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(churned_domains['domain'][::-1], churned_domains['churned_users'][::-1],
        color='#E87B4C', edgecolor='white')
ax.set_xlabel('Churned Users')
ax.set_title('Top Email Domains – Churned Users', fontweight='bold')
plt.tight_layout()
plt.savefig('data/top_churned_domains.png', bbox_inches='tight')
plt.show()


## 4. Predictive Model – Churn Risk Scoring

We train two models:
1. **Logistic Regression** – interpretable baseline  
2. **Random Forest** – higher performance

Features used: days since first seen, days since last contacted, engagement window, web sessions, whether user ever replied, and free-email flag.


In [ ]:
FEATURES = [
    'days_since_first_seen',
    'days_since_last_contacted',
    'engagement_window_days',
    'web_sessions',
    'has_been_heard_from',
    'is_free_email',
]

model_df = df[FEATURES + ['churned']].dropna()
X = model_df[FEATURES]
y = model_df['churned']

print(f"Dataset for modelling: {len(model_df):,} rows | Churn rate: {y.mean():.1%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ── Logistic Regression
lr_pipe = Pipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
lr_pipe.fit(X_train, y_train)

# ── Random Forest
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, max_depth=6)
rf.fit(X_train, y_train)

# ── Cross-val AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_auc = cross_val_score(lr_pipe, X, y, cv=cv, scoring='roc_auc').mean()
rf_auc = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc').mean()

print(f"\nLogistic Regression  – 5-fold CV AUC: {lr_auc:.3f}")
print(f"Random Forest        – 5-fold CV AUC: {rf_auc:.3f}")


### 4a. Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix (Random Forest)
y_pred_rf = rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active','Churned'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Random Forest – Confusion Matrix', fontweight='bold')

# ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_pipe.predict_proba(X_test)[:,1])
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf.predict_proba(X_test)[:,1])
auc_lr = roc_auc_score(y_test, lr_pipe.predict_proba(X_test)[:,1])
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Reg. (AUC={auc_lr:.2f})', color='#4C9BE8')
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.2f})', color='#E87B4C')
axes[1].plot([0,1],[0,1], 'k--', linewidth=0.8, label='Random baseline')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/model_evaluation.png', bbox_inches='tight')
plt.show()

print("\nClassification Report – Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=['Active','Churned']))


### 4b. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot.barh(ax=ax, color='#4C9BE8', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest – Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('data/feature_importance.png', bbox_inches='tight')
plt.show()


## 5. Churn Risk Scoring

Assign every user a **churn probability** using the Random Forest, then bucket them into risk tiers for the CSR team to action.


In [ ]:
# Score all users (fill any remaining NaN conservatively)
X_all = df[FEATURES].fillna({
    'days_since_last_contacted': 999,
    'engagement_window_days': 0,
})
df['churn_probability'] = rf.predict_proba(X_all)[:,1]

def risk_tier(p):
    if p >= 0.6:   return 'High Risk'
    elif p >= 0.3: return 'Medium Risk'
    else:          return 'Low Risk'

df['risk_tier'] = df['churn_probability'].apply(risk_tier)

tier_counts = df['risk_tier'].value_counts().reindex(['High Risk','Medium Risk','Low Risk'])
print("Risk tier distribution:")
print(tier_counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors_tier = {'High Risk':'#E87B4C', 'Medium Risk':'#F5C542', 'Low Risk':'#4C9BE8'}
bars = ax.bar(tier_counts.index, tier_counts.values,
              color=[colors_tier[t] for t in tier_counts.index],
              edgecolor='white', width=0.5)
ax.set_ylabel('Number of Users')
ax.set_title('Users by Churn Risk Tier', fontweight='bold')
for bar, v in zip(bars, tier_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('data/risk_tiers.png', bbox_inches='tight')
plt.show()


## 6. Export – High-Risk Users for CSR Outreach

In [ ]:
high_risk = (df[df['risk_tier']=='High Risk']
             [['Email','email_domain','days_since_first_seen',
               'days_since_last_contacted','has_been_heard_from',
               'is_free_email','churn_probability','risk_tier']]
             .sort_values('churn_probability', ascending=False))

print(f"High-risk users: {len(high_risk)}")
high_risk.to_csv('data/high_risk_users.csv', index=False)
print("Saved to data/high_risk_users.csv")
high_risk.head(10)


## 7. Summary & Recommendations

### Key Findings

| Finding | Detail |
|---------|--------|
| Overall churn rate | ~2.4% (55 of 2,286 users) |
| Strongest retention signal | Users who replied at least once had **0% churn** |
| Highest risk segment | Free email users (Gmail/Hotmail) churn at **2×** the rate of business email users |
| Model performance | Random Forest evaluated with 5-fold cross-validated AUC (see Section 4) |

### Recommendations for CSR Team

1. **Prioritise reply-triggering outreach** — a single response from a user predicts full retention. Design campaigns that invite a reply (questions, surveys, offers).
2. **Flag free-email signups early** — assign a CSR owner within the first week.
3. **Use the risk-tier export** (`high_risk_users.csv`) to focus retention effort on High Risk users before they unsubscribe.
4. **Collect richer data going forward** — plan name, login frequency, and feature usage would significantly improve model accuracy.

### Limitations & Future Work

- Dataset is largely a bulk import (Feb 2026), limiting temporal diversity.
- Adding behavioural signals (feature usage, login events) would improve the model.
- A/B test retention campaigns on High Risk vs Low Risk segments to validate the model's business impact.
